# Listener Prior (Dual Dataset) - Keyterms Prediction (Public Repo, Colab GPU)

This notebook:
- clones the repo into `/content/listener-prior`
- installs dependencies (pins `datasets<4.0.0` so MultiWOZ/DailyDialog script datasets load)
- trains the bi-encoder to predict the **other speaker's next utterance**
- builds **keyterms/keywords** from retrieved candidates (for Deepgram STT injection)
- writes outputs to Google Drive so they persist

In [ ]:
# --- CONFIG: set your GitHub repo here ---
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/<YOUR_REPO_NAME>.git"  # <- edit
PROJECT_DIR = "/content/listener-prior"

In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!ls -la

In [ ]:
# Install deps. We exclude packages that Colab already provides to avoid conflicts.
# Colab has: torch, torchvision, torchaudio, pandas, numpy, requests, packaging, etc.
import pathlib

req = pathlib.Path("requirements.txt").read_text().splitlines()
# Exclude packages that Colab pre-installs and causes conflicts
exclude_pkgs = {"torch", "pandas", "numpy", "requests", "packaging"}
req_filtered = [
    r for r in req 
    if r.strip() and not any(r.strip().startswith(ex) or r.strip().startswith(f"{ex}==") or r.strip().startswith(f"{ex}>=") or r.strip().startswith(f"{ex}<=") or r.strip().startswith(f"{ex}~=") or r.strip().startswith(f"{ex}!=") for ex in exclude_pkgs)
]
pathlib.Path("/tmp/requirements_colab.txt").write_text("\n".join(req_filtered) + "\n")

print("Filtered requirements (excluding Colab pre-installed packages):")
print("\n".join(req_filtered[:10]))
print("...")

!python -m pip install -U pip
!python -m pip install -r /tmp/requirements_colab.txt --upgrade --force-reinstall

import datasets
print("\ndatasets:", datasets.__version__)
assert tuple(int(x) for x in datasets.__version__.split(".")[:1]) < (4,), "datasets must be < 4.0.0; restart runtime after install"

In [ ]:
# If the assert above failed, go to Runtime -> Restart runtime, then rerun from the top.
# Verify post-install versions
import importlib.metadata as _md
def _ver(pkg: str):
    try:
        return _md.version(pkg)
    except Exception:
        return None

print("Post-install versions:")
print(f"  datasets: {_ver('datasets')}")
print(f"  sentence-transformers: {_ver('sentence-transformers')}")
print(f"  transformers: {_ver('transformers')}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hugging Face token (recommended).
# In Colab: Tools -> Secrets -> add HF_TOKEN.
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

# Clear any stale/expired tokens that might already exist in the environment.
for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found. Public datasets/models should still work without it.')

In [ ]:
import datetime
run_id = datetime.datetime.now().strftime('dual_keyterms_%Y%m%d_%H%M%S')
OUTPUT_DIR = f"/content/drive/MyDrive/listener_prior_runs/{run_id}"
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Train. This evaluates on TEST each epoch and updates encoder_best/ when improved.
# target_role=SYSTEM means: predict what the other person will say next.
!python scripts/train_dual_epoch_test.py \
  --output_dir "$OUTPUT_DIR" \
  --device auto \
  --epochs 6 \
  --batch_size 32 \
  --learning_rate 4.3e-5 \
  --weight_decay 0.01 \
  --adam_beta1 0.95 \
  --adam_beta2 0.98 \
  --adam_eps 1e-8 \
  --grad_accum_steps 2 \
  --warmup_ratio 0.0 \
  --history_turns 6 \
  --target_role SYSTEM \
  --val_ratio 0.05 \
  --test_ratio 0.15 \
  --max_dialogs_multiwoz 0 \
  --max_dialogs_dailydialog 0

In [ ]:
# Build keyterms/keywords from retrieved candidates.
# This uses the trained model and index to produce a prior for a sample history.
RUN_DIR = OUTPUT_DIR
!python -m src.demo_offline --run "$RUN_DIR" --encoder_subdir encoder_best --topk 10